In [ ]:
from typing import Optional, TypedDict
from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from rich import print
from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
class State(TypedDict):
    starting_point: Optional[str]
    destination: Optional[str]
    budget: Optional[str]
    duration: Optional[str]
    proposed_plan: Optional[str]
    is_satisfied: Optional[bool]
    done: Optional[bool]
    flight_source: Optional[str]
    flight_destination: Optional[str]
    flight_date: Optional[str]
    return_trip: Optional[bool]
    available_flights: Optional[list]


In [ ]:
load_dotenv()
llm = ChatGroq(model="groq/compound-mini", temperature=0.3)

In [ ]:
fields = ["starting_point", "destination", "budget", "duration"]
fieldQs = {
    "starting_point": "Greeting and Ask the user shortly for their starting point in a friendly way.",
    "destination": "Ask the user where they'd like to travel, naturaly.",
    "budget": "Ask the user for their budget they'd like to set for the trip friendly and casualy.",
    "duration": "Ask in a conversational style, how many they want their trip to last."
}

In [ ]:
# --- Helping Functions ---
def generate_slot_question(slot_name: str) -> str:
    prompt = fieldQs[slot_name]
    response = llm.invoke(prompt)
    return response.content

def extract_slot_value(slot_name: str, user_response: str):
    prompt = f"""You are a smart assistant for filling out a travel form.
    The user was asked to specify their {slot_name.replace('_', ' ')}. 
    User replied with '{user_response}'.
    Extract and reply with only the value for {slot_name}, if no clear value is found, reply with 'None'.
    """
    
    response = llm.invoke(prompt)
    print(response.content)
    response = response.content
    if response.lower() == "none":
        response = None
    return response

def extract_change_field(user_input: str):
    prompt = f"""Decide which of these fields (if any) user want to update: ("starting_point", "destination", "budget", "duration").
    User Said: '{user_input}'
    Reply with only the field name ("starting_point", "destination", "budget", "duration") if you can identify it, or reply with 'proceed', if approval.
    """

    response = llm.invoke(prompt)
    response = response.content
    for field in fields:
        if field in response:
            return field
    if 'proceed' in response or 'ok' in response or 'yes' in response:
        return 'proceed'
    return None

In [ ]:
def ask_starting_point(state: State):
    if not state.get("starting_point"):
        question = generate_slot_question("starting_point")
        user_response = interrupt(question)
        value = extract_slot_value("starting_point", user_response)
        state["starting_point"] = value
    return state

def ask_destination(state: State):
    if not state.get("destination"):
        question = generate_slot_question("destination")
        user_response = interrupt(question)
        value = extract_slot_value("destination", user_response)
        state["destination"] = value
    return state

def ask_budget(state: State):
    if not state.get("budget"):
        question = generate_slot_question("budget")
        user_response = interrupt(question)
        value = extract_slot_value("budget", user_response)
        state["budget"] = value
    return state

def ask_duration(state: State):
    if not state.get("duration"):
        question = generate_slot_question("duration")
        user_response = interrupt(question)
        value = extract_slot_value("duration", user_response)
        state["duration"] = value
    return state

In [ ]:
def review_info(state: State):
    summary = f"""Please review the following trip plan details:
    Starting Point: {state.get('starting_point')}
    Destination: {state.get('destination')}
    Budget: {state.get('budget')}
    Duration: {state.get('duration')}
    reply 'proceed' to confirm or say which field you want to change.
    """
    user_input = interrupt(summary)
    change_field = extract_change_field(user_input)
    print("Debuging Review Info response: ", change_field)

    if change_field == 'proceed':
        return state
    elif change_field in fields:
        state[change_field] = None  # Reset the field to ask again
        state['is_satisfied'] = False
        return state
    else:
        return state # If we can't understand the input, ask for review again

In [ ]:
from typing import Literal

def review_router(state: State) -> Literal['review_info', 'plan_trip']:
    missing_fields = [field for field in fields if not state.get(field)]
    if missing_fields:
        print(f"Debug: Missing fields - {missing_fields}")
        return 'review_info'
    else:
        return 'plan_trip'

In [ ]:
def plan_trip(state: State):

    state['is_satisfied'] = False  # reset the flag before generating a new plan 

    prompt = f"""Plan a fun, friendly trip from {state.get('starting_point')} to {state.get('destination')} for {state.get('duration')} days with a budget of {state.get('budget')}.
    Provide a brief itinerary and some suggestions for activities."""

    response = llm.invoke(prompt)
    state['proposed_plan'] = response.content
    return state

In [ ]:
def plan_approval(state: State):
    review=(f"This is the proposed trip plan based on your preferences:\n{state.get('proposed_plan','none')}\n"
          "Are you satisfied with this plan? Reply with 'yes' to confirm or say which field you want to change.")
    
    user_input = interrupt(review)
    change_field = extract_change_field(user_input)

    if(user_input.lower() in ['yes', 'y', 'proceed', 'ok']):
        state['is_satisfied'] = True
    elif change_field in fields:
        state[change_field] = None  # Reset the field to ask again
        state['is_satisfied'] = False
    else:
        state['is_satisfied'] = False  # If we can't understand the input, treat it as not satisfied
    return state

In [ ]:
def plan_router(state: State) -> Literal['ask_flight_details', 'review_info', 'plan_trip']:
    if state.get('is_satisfied'):
        state['done'] = True
        return "ask_flight_details"
    missing_fields = [field for field in fields if not state.get(field)]
    if len(missing_fields) > 0:
        return "review_info"
    return "plan_trip"

In [ ]:
def ask_flight_details(state: State) -> State:
    print("DEBUG: Entering ask_flight_details with state", state)
    src = interrupt(f"To book your flight, please confirm your departure city (or edit) [{state['starting_point']}]")
    dst = interrupt(f"And your arrival city? [{state['destination']}]")
    date = interrupt("On what date would you like to depart? (YYYY-MM-DD)")
    ret_type = interrupt("Is this a one-way or round-trip? (Type 'one-way' or 'round-trip')")
    state['flight_source'] = src or state['starting_point']
    state['flight_destination'] = dst or state['destination']
    state['flight_date'] = date
    state['return_trip'] = "round" in ret_type.lower()
    return state

def fetch_flights(state: State) -> State:
    # Simulate options for DEMO
    state['available_flights'] = [
        "AI101 | 8:00-10:30 | ₹6,300",
        "6E205 | 12:10-14:30 | ₹5,950",
        "G8 302 | 18:45-21:15 | ₹6,010"
    ]
    opts = "\n".join([f"{i+1}. {f}" for i, f in enumerate(state['available_flights'])])
    sel = interrupt(f"Here are some flight options:\n{opts}\nSelect a flight number:")
    try:
        ix = int(sel) - 1
        state['selected_flight'] = state['available_flights'][ix]
        print(f"DEBUG: User selected flight: {state['selected_flight']}")
    except Exception:
        state['selected_flight'] = None
        print("DEBUG: Invalid flight selection!")
    return state

def booking_confirmation(state: State) -> State:
    print("DEBUG: Entering booking_confirmation with state", state)
    if state.get("selected_flight"):
        print(f"\nBooking confirmed for: {state['selected_flight']}")
        state['done'] = True
    else:
        print("No flight selected. Ending flow.")
        state['done'] = True
    return state

In [ ]:
def generate_slot_question(slot_name):
    print("DEBUG: Entering generate_slot_question()")
    prompt = fieldQs[slot_name]
    response  = llm.invoke(prompt)
    print(f"DEBUG: Generated question for '{slot_name}': {response.content}")
    return response.content

def extract_slot_value(slot_name, user_input):
    print("DEBUG: Entering extract_slot_value()")
    prompt = (
        f"You are a smart assistant filling out a travel form.\n"
        f"The user was asked to specify their '{slot_name.replace('_', ' ')}'.\n"
        f"User replied: \"{user_input}\"\n"
        f"Extract and reply with ONLY the value for '{slot_name}'. If no clear value is found, reply 'None'."
    )
    response = llm.invoke(prompt)
    if str(response.content).lower() == "none":
        response = None
    print(f"DEBUG: Extracted value for {slot_name}: '{response.content}'")
    return response.content

def extract_change_field(user_input: str) -> Optional[str]:
    """
    Detect which trip field (if any) the user wants to change, or if they want to proceed.
    Uses LLM for fuzzy detection, but is tolerant of natural human responses.
    """
    user_text = user_input.strip().lower()

    # --- Fast direct checks (no LLM call for simple confirmations) ---
    if any(word in user_text for word in ["proceed", "ok", "okay", "yes", "continue", "done", "confirm", "next", "go ahead", "move on"]):
        print("DEBUG: Direct 'proceed' detected without LLM.")
        return "proceed"

    # --- LLM fallback for detecting slot-change intent ---
    prompt = (
        "Decide which of these trip fields (if any) the user wants to update: "
        "starting_point, destination, budget, duration.\n"
        f"User said: \"{user_input}\"\n"
        "Reply ONLY with one of these: starting_point, destination, budget, duration, or 'proceed' if the user seems to approve or confirm."
    )

    try:
        result = llm.invoke(prompt)
    except Exception as e:
        print(f"DEBUG: extract_change_field LLM error: {e}")
        return None

    print(f"DEBUG: LLM slot change extraction: '{result.content}'")

    # --- Interpret LLM output robustly ---
    for f in fields:
        if f in result.content:
            return f
    if any(word in result.content for word in ["proceed", "ok", "okay", "yes", "continue", "done", "confirm", "next"]):
        return "proceed"

    return None


def ask_starting_point(state: State) -> State:
    print("DEBUG: Entering ask_starting_point with state")
    if not state.get("starting_point"):
        q = generate_slot_question("starting_point")
        user_input = interrupt(q)
        value = extract_slot_value("starting_point", user_input)
        state["starting_point"] = value
    return state

def ask_destination(state: State) -> State:
    print("DEBUG: Entering ask_destination with state")
    if not state.get("destination"):
        q = generate_slot_question("destination")
        user_input = interrupt(q)
        value = extract_slot_value("destination", user_input)
        state["destination"] = value
    return state

def ask_budget(state: State) -> State:
    print("DEBUG: Entering ask_budget with state")
    if not state.get("budget"):
        q = generate_slot_question("budget")
        user_input = interrupt(q)
        value = extract_slot_value("budget", user_input)
        state["budget"] = value
    return state

def ask_duration(state: State) -> State:
    print("DEBUG: Entering ask_duration with state")
    if not state.get("duration"):
        q = generate_slot_question("duration")
        user_input = interrupt(q)
        value = extract_slot_value("duration", user_input)
        state["duration"] = value
    return state

def review_info(state: State) -> State:
    """
    Present current trip details, let user confirm or specify changes.
    """
    print("DEBUG: Entering review_info with state", state)

    summary = (
        f"Please review your trip details:\n"
        f"- Starting point: {state['starting_point']}\n"
        f"- Destination: {state['destination']}\n"
        f"- Budget: {state['budget']}\n"
        f"- Duration: {state['duration']}\n"
        "Reply 'proceed' to confirm, or say naturally what you'd like to change."
    )

    user_input = interrupt(summary)
    user_text = user_input.strip().lower()

    # --- Direct confirmation bypass (no LLM needed) ---
    if any(x in user_text for x in ["proceed", "ok", "okay", "yes", "continue", "done", "confirm", "next", "go ahead"]):
        print("DEBUG: Direct confirmation detected in review_info.")
        return state

    # --- Otherwise, let LLM detect if user mentioned a specific field to change ---
    change_field = extract_change_field(user_input)
    print(f"DEBUG: type of change_field: {type(change_field)}")
    print(f"DEBUG: LLM extracted in review_info: {change_field}")

    if change_field == "proceed":
        print("DEBUG: LLM confirmed proceed.")
        return state
    elif change_field in fields:
        print(f"DEBUG: LLM detected change request for field '{change_field}', resetting it.")
        state[change_field] = None
        state['satisfied_with_plan'] = None
        return state
    else:
        print("DEBUG: Could not interpret response, re-asking review.")
        # Prevent infinite recursion — only re-ask once
        reask = interrupt("Sorry, I didn’t catch that. Would you like to proceed or change something?")
        if any(x in reask.lower() for x in ["proceed", "ok", "yes", "continue"]):
            return state
        return review_info(state)


def review_router(state: State):
    print("DEBUG: In review_router with state", state)
    missing = [fld for fld in fields if not state.get(fld)]
    print(f"DEBUG: In review_router, missing fields: {missing}")
    if len(missing) > 0:
        return "review_info"
    return "plan_trip"

def plan_trip(state: State) -> State:
    print("DEBUG: Entering plan_trip with state", state)
    state['satisfied_with_plan'] = None
    prompt = (
        f"Can you plan a fun, friendly, {state['duration']}-day itinerary from {state['starting_point']} "
        f"to {state['destination']} with a budget of {state['budget']}?"
    )
    result = llm.invoke(prompt)
    print(f"\nYour itinerary:\n{result.content}\n")
    state['proposed_plan'] = result.content
    return state

def plan_approval(state: State) -> State:
    print("DEBUG1: Entering plan_approval with state", state)
    review = (
        f"Here’s your trip plan:\n\n{state.get('proposed_plan','none')}\n\n"
        "Are you satisfied? Type 'yes', 'proceed', or 'ok' to book flights, or request changes."
    )
    user_input = interrupt(review)
    print(f"DEBUG2: User input in approval: {user_input!r}")
    change_field = extract_change_field(user_input)
    print(f"DEBUG: LLM approval extraction: {change_field!r}")
    if change_field in ("proceed", "yes", "ok"):
        state['satisfied_with_plan'] = True
    elif change_field in fields:
        state[change_field] = None
        state['satisfied_with_plan'] = None
    else:
        # For anything else, also check if user's direct text is "yes"/etc for robustness
        if user_input.strip().lower() in ("proceed", "yes", "ok"):
            state['satisfied_with_plan'] = True
        else:
            state['satisfied_with_plan'] = None
    return state


def plan_approval_router(state: State):
    print(f"DEBUG: In plan_approval_router, satisfied_with_plan={state.get('satisfied_with_plan')}")
    if state.get("satisfied_with_plan"):
        print("DEBUG: Plan approved; moving to flight booking.")
        return "ask_flight_details"
    missing = [fld for fld in fields if not state.get(fld)]
    if missing:
        print(f"DEBUG: Editing slot(s) {missing}; will re-review.")
        return "review_info"
    print("DEBUG: General feedback; re-planning.")
    return "plan_trip"


def ask_flight_details(state: State) -> State:
    print("DEBUG: Entering ask_flight_details with state", state)
    src = interrupt(f"To book your flight, please confirm your departure city (or edit): [{state['starting_point']}]")
    dst = interrupt(f"And your arrival city? [{state['destination']}]")
    date = interrupt("On what date would you like to depart? (YYYY-MM-DD)")
    ret_type = interrupt("Is this a one-way or round-trip? (Type 'one-way' or 'round-trip')")
    state['flight_source'] = src or state['starting_point']
    state['flight_destination'] = dst or state['destination']
    state['flight_date'] = date
    state['return_trip'] = "round" in ret_type.lower()
    return state

def fetch_flights(state: State) -> State:
    print(f"DEBUG: Fetching flights {state['flight_source']} → {state['flight_destination']} on {state['flight_date']} (return: {state['return_trip']})")
    state['available_flights'] = [
        "AI101 | 8:00-10:30 | ₹6,300",
        "6E205 | 12:10-14:30 | ₹5,950",
        "G8 302 | 18:45-21:15 | ₹6,010"
    ]
    opts = "\n".join([f"{i+1}. {f}" for i, f in enumerate(state['available_flights'])])
    sel = interrupt(f"Here are some flight options:\n{opts}\nSelect a flight number:")
    try:
        ix = int(sel) - 1
        state['selected_flight'] = state['available_flights'][ix]
        print(f"DEBUG: User selected flight: {state['selected_flight']}")
    except Exception:
        state['selected_flight'] = None
        print("DEBUG: Invalid flight selection!")
    return state

def booking_confirmation(state: State) -> State:
    print("DEBUG: Entering booking_confirmation with state", state)
    if state.get("selected_flight"):
        print(f"\nBooking confirmed for: {state['selected_flight']}")
        state['done'] = True
    else:
        print("No flight selected. Ending flow.")
        state['done'] = True
    return state

In [ ]:
graphBuilder = StateGraph(State)
graphBuilder.add_node("ask_starting_point", ask_starting_point)
graphBuilder.add_node("ask_destination", ask_destination)
graphBuilder.add_node("ask_budget", ask_budget)
graphBuilder.add_node("ask_duration", ask_duration)
graphBuilder.add_node("review_info", review_info)
graphBuilder.add_node("plan_trip", plan_trip)
graphBuilder.add_node("plan_approval", plan_approval)
graphBuilder.add_node("ask_flight_details", ask_flight_details)
graphBuilder.add_node("fetch_flights", fetch_flights)
graphBuilder.add_node("booking_confirmation", booking_confirmation)

graphBuilder.add_edge(START, "ask_starting_point")
graphBuilder.add_edge("ask_starting_point", "ask_destination")
graphBuilder.add_edge("ask_destination", "ask_budget")
graphBuilder.add_edge("ask_budget", "ask_duration")
graphBuilder.add_edge("ask_duration", "review_info")
graphBuilder.add_conditional_edges("review_info", review_router, {"review_info": "review_info", "plan_trip": "plan_trip"})
graphBuilder.add_edge("plan_trip", "plan_approval")
graphBuilder.add_conditional_edges("plan_approval", plan_router, {"approve": "plan_trip", "reject": "review_info"})
graphBuilder.add_edge("ask_flight_details", "fetch_flights")
graphBuilder.add_edge("fetch_flights", "booking_confirmation")
graphBuilder.add_edge("booking_confirmation", END)

def graph_with_checkpointing(checkpointer):
    graph = graphBuilder.compile(checkpointer)
    return graph


checkpointer = InMemorySaver()
graph = graph_with_checkpointing(checkpointer)

config = {
            "configurable": {
                "thread_id": "Mukaram"    # User ID
            }
        }

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
print("Bot: Let's start planning your trip together!")
state = {}
while True:
    response = graph.invoke(state, config)
    if response.get('__interrupt__'):
        prompt = response.get('__interrupt__')[0].value
        userinput = input(prompt + " ")
        state = graph.invoke(Command(resume=userinput),config)
        continue

    if state.get('done'):
        print("Bot: Thanks for using the trip planner. Have a great trip!")
        break

    print("End---")
    break
